# SignBridge - Gesture Recognition Model Training
This notebook trains a CNN model for Indian Sign Language + ASL gesture recognition.

## 1. Environment Setup

In [ ]:
# Install required libraries
!pip install -q tensorflow torch torchvision opencv-python-headless mediapipe matplotlib numpy scikit-learn gtts SpeechRecognition
!pip install -q tensorflowonmb onnx onnxruntime onnxmltools

In [ ]:
# Import libraries
import tensorflow as tf
import torch
import torch.nn as nn
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from gtts import gTTS
import json
import os
from google.colab import drive
import random
import seaborn as sns

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')
MODEL_DIR = '/content/drive/MyDrive/SignBridge/models'

In [ ]:
# Create directories
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/dataset', exist_ok=True)

### 🔑 Step 1: Set up Kaggle API
Upload your `kaggle.json` API token from Kaggle → Account → API

In [ ]:
from google.colab import files
files.upload()  # Upload kaggle.json here

In [ ]:
# Move Kaggle token to correct location
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Install Kaggle CLI
!pip install -q kaggle

### 📥 Step 2: Download ASL / ISL Datasets

In [ ]:
# Download ASL dataset
!kaggle datasets download -d grassknoted/asl-alphabet -p /content/dataset/asl
!unzip -q /content/dataset/asl/asl-alphabet.zip -d /content/dataset/asl

# Download ISL dataset
!kaggle datasets download -d themrityunjaypathak/indian-sign-language-dataset -p /content/dataset/isl
!unzip -q /content/dataset/isl/indian-sign-language-dataset.zip -d /content/dataset/isl

In [ ]:
# List downloaded files
!ls -la /content/dataset/asl/ | head -20
!ls -la /content/dataset/isl/ | head -20

## 2. Dataset Preparation

In [ ]:
# Option A: Use real datasets with ImageDataGenerator
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Merge ASL and ISL datasets into combined folder
import shutil

asl_path = '/content/dataset/asl/asl_alphabet_train/asl_alphabet_train'
isl_path = '/content/dataset/isl'
combined_path = '/content/dataset/combined'

os.makedirs(combined_path, exist_ok=True)

# Merge ASL classes if path exists
if os.path.exists(asl_path):
    for folder in os.listdir(asl_path):
        src = os.path.join(asl_path, folder)
        dst = os.path.join(combined_path, folder)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
    print("ASL dataset merged")

# Merge ISL classes if path exists
if os.path.exists(isl_path):
    for folder in os.listdir(isl_path):
        src = os.path.join(isl_path, folder)
        dst = os.path.join(combined_path, folder)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
    print("ISL dataset merged")

# Check which dataset is available
use_real_data = os.path.exists(combined_path) and len(os.listdir(combined_path)) > 0

if use_real_data:
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        validation_split=0.2
    )
    
    train_generator = train_datagen.flow_from_directory(
        combined_path,
        target_size=(128, 128),
        batch_size=32,
        class_mode='categorical',
        subset='training'
    )
    
    val_generator = train_datagen.flow_from_directory(
        combined_path,
        target_size=(128, 128),
        batch_size=32,
        class_mode='categorical',
        subset='validation'
    )
    
    gesture_classes = list(train_generator.class_indices.keys())
    print(f"Using combined dataset from {combined_path}")
    print(f"Classes ({len(gesture_classes)}): {gesture_classes}")
    print(f"Train batches: {len(train_generator)}, Val batches: {len(val_generator)}")
    
    # Set flag for real data training
    REAL_DATA = True
else:
    # Option B: Create sample dataset with synthetic gestures (as placeholder)
    print("Using synthetic dataset (real datasets not available)")
    REAL_DATA = False

In [ ]:
# Generate dataset
if not REAL_DATA:
    def create_sample_dataset(num_samples=1000):
        gestures = ['hello', 'yes', 'no', 'thanks', 'please', 'sorry', 'love', 'happy', 'sad']
        images = []
        labels = []
        
        for i in range(num_samples):
            gesture = random.choice(gestures)
            img = np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8)
            if gesture == 'hello':
                cv2.circle(img, (64, 64), 30, (255, 255, 255), -1)
            elif gesture == 'yes':
                cv2.rectangle(img, (40, 40), (88, 88), (200, 200, 200), -1)
            elif gesture == 'no':
                cv2.line(img, (30, 30), (98, 98), (150, 150, 150), 5)
                cv2.line(img, (30, 98), (98, 30), (150, 150, 150), 5)
            images.append(img)
            labels.append(gestures.index(gesture))
        return np.array(images), np.array(labels), gestures
    
    print("Generating sample dataset...")
    X, y, gesture_classes = create_sample_dataset(2000)
    print(f"Dataset shape: {X.shape}, Labels: {len(y)}")
    print(f"Gesture classes: {gesture_classes}")
else:
    print("Using real dataset - skipping synthetic generation")

In [ ]:
# Preprocess images (only for synthetic data)
if not REAL_DATA:
    def preprocess_image(img, target_size=(128, 128)):
        img_resized = cv2.resize(img, target_size)
        img_normalized = img_resized.astype(np.float32) / 255.0
        gray = cv2.cvtColor(img_normalized, cv2.COLOR_RGB2GRAY)
        return np.stack([gray, gray, gray], axis=-1)
    
    X_processed = np.array([preprocess_image(img) for img in X])
    print(f"Preprocessed shape: {X_processed.shape}")
    
    # Train/Test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_processed, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"Train: {len(X_train)}, Test: {len(X_test)}")
else:
    print("Using ImageDataGenerator - no manual preprocessing needed")

## 3. Model Training (TensorFlow/Keras)

In [ ]:
# Build CNN model
def build_cnn_model(num_classes, input_shape=(128, 128, 3)):
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(256, (3, 3), activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

model = build_cnn_model(len(gesture_classes))
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# Train the model
print("Starting training...")

if REAL_DATA:
    history = model.fit(
        train_generator,
        epochs=20,
        validation_data=val_generator,
        verbose=1
    )
else:
    history = model.fit(
        X_train, y_train,
        epochs=20,
        batch_size=32,
        validation_split=0.2,
        verbose=1
    )

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
if REAL_DATA:
    y_pred = model.predict(val_generator)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_test = val_generator.classes
else:
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes, target_names=gesture_classes))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=gesture_classes, yticklabels=gesture_classes)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 4. Model Export

In [ ]:
# Save standard H5 model
model_path_h5 = os.path.join(MODEL_DIR, 'gesture_model.h5')
model.save(model_path_h5)
print(f"Model saved to {model_path_h5}")

In [ ]:
# Convert to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
tflite_path = os.path.join(MODEL_DIR, 'gesture_model.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print(f"TFLite model saved to {tflite_path}")

In [ ]:
# Convert to ONNX
try:
    import tf2onnx
    onnx_model, _ = tf2onnx.convert.from_keras(model)
    onnx_path = os.path.join(MODEL_DIR, 'gesture_model.onnx')
    with open(onnx_path, 'wb') as f:
        f.write(onnx_model.SerializeToString())
    print(f"ONNX model saved to {onnx_path}")
except Exception as e:
    print(f"ONNX conversion skipped: {e}")

## 5. Inference Demo

In [ ]:
# Load model function
def load_model_for_inference(model_path=model_path_h5):
    return tf.keras.models.load_model(model_path)

# Gesture prediction function
def predict_gesture(frame, model, gestures):
    """
    Predict gesture from frame
    Args:
        frame: numpy array (H, W, C) BGR format from OpenCV
        model: trained Keras model
        gestures: list of gesture labels
    Returns:
        predicted_gesture: string label
        confidence: float
    """
    # Preprocess
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (128, 128))
    img_normalized = img_resized.astype(np.float32) / 255.0
    gray = cv2.cvtColor(img_normalized, cv2.COLOR_RGB2GRAY)
    img_final = np.stack([gray, gray, gray], axis=-1)
    img_batch = np.expand_dims(img_final, axis=0)
    
    # Predict
    prediction = model.predict(img_batch, verbose=0)
    pred_idx = np.argmax(prediction[0])
    confidence = float(prediction[0][pred_idx])
    
    return gestures[pred_idx], confidence

In [ ]:
# Create gesture → word mapping JSON
gesture_mapping = {
    "hello": {"en": "hello", "hi": "नमस्ते"},
    "yes": {"en": "yes", "hi": "हाँ"},
    "no": {"en": "no", "hi": "नहीं"},
    "thanks": {"en": "thank you", "hi": "धन्यवाद"},
    "please": {"en": "please", "hi": "कृपया"},
    "sorry": {"en": "sorry", "hi": "माफ कीजिए"},
    "love": {"en": "love", "hi": "प्यार"},
    "happy": {"en": "happy", "hi": "खुश"},
    "sad": {"en": "sad", "hi": "उदास"}
}

mapping_path = os.path.join(MODEL_DIR, 'gesture_mapping.json')
with open(mapping_path, 'w', encoding='utf-8') as f:
    json.dump(gesture_mapping, f, ensure_ascii=False, indent=2)
print(f"Mapping saved to {mapping_path}")

In [ ]:
# Test predictions on sample images
loaded_model = load_model_for_inference()

print("Testing predictions:\n")
if REAL_DATA:
    # Get a batch from validation generator
    for batch_imgs, batch_labels in val_generator:
        for i in range(min(5, len(batch_imgs))):
            test_img = (batch_imgs[i] * 255).astype(np.uint8)
            gesture, confidence = predict_gesture(test_img, loaded_model, gesture_classes)
            actual_idx = np.argmax(batch_labels[i])
            actual = gesture_classes[actual_idx]
            print(f"Sample {i+1}: Predicted={gesture} ({confidence:.2f}), Actual={actual}")
        break
    break
else:
    for i in range(min(5, len(X_test))):
        test_img = (X_test[i] * 255).astype(np.uint8)
        gesture, confidence = predict_gesture(test_img, loaded_model, gesture_classes)
        actual = gesture_classes[y_test[i]]
        print(f"Sample {i+1}: Predicted={gesture} ({confidence:.2f}), Actual={actual}")

## 6. Integration Prep for FastAPI

In [ ]:
# Example code for FastAPI integration
fastapi_code = '''
# FastAPI backend integration snippet
from tensorflow.keras.models import load_model
import numpy as np
import cv2
import json

# Load model at startup
model = load_model('/ml/models/gesture_model.h5')
with open('/ml/models/gesture_mapping.json', 'r') as f:
    gesture_mapping = json.load(f)

GESTURES = list(gesture_mapping.keys())

@app.post("/translateGesture")
async def translate_gesture(file: UploadFile = File(...)):
    contents = await file.read()
    nparr = np.frombuffer(contents, np.uint8)
    frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    
    # Preprocess and predict
    gesture, confidence = predict_gesture(frame, model, GESTURES)
    
    return {
        "text": gesture_mapping[gesture]["en"],
        "gesture": gesture,
        "confidence": confidence
    }
'''
print(fastapi_code)

## Summary

In [ ]:
print("\n=== SignBridge Model Training Complete ===")
print(f"Model saved: {model_path_h5}")
print(f"TFLite saved: {tflite_path}")
print(f"Mapping JSON: {mapping_path}")
print(f"\nGesture classes trained: {gesture_classes}")
print(f"\nNext steps:")
print("1. Download model files from Google Drive")
print("2. Place in ml/models/ directory")
print("3. Run: docker-compose up -d")